# 🎮 AI Game Dev — GPU Reinforcement Learning Training

Trains the AI to produce **better games** through trial, error, rewards and penalties — on a **free Colab GPU** (T4/A100).

### How it works
| Phase | What happens |
|-------|--------------|
| 1 | AI generates pixel art, GDScript, game designs via PixelLab + LLM |
| 2 | Evaluator scores every output (0–100) on Art / Code / Design axes |
| 3 | PPO policy gradient updates agent weights toward higher scores |
| 4 | ONNX checkpoint exported, pushed back to GitHub |

### Curriculum levels
| Level | Min Score | Tasks |
|-------|-----------|-------|
| 1 Basics | 0+ | Simple characters, tiles, 20-line scripts |
| 2 Intermediate | 40+ | Animated chars, full scenes, enemy AI |
| 3 Advanced | 65+ | Complete game scenes, systems |
| 4 Expert | 80+ | Full polished isekai RPG |

---
**Before running**: set your secrets in the cell below (or use Colab Secrets via the 🔑 icon).

In [ ]:
# ── AUTO-RECONNECT: keeps Colab alive (run this cell FIRST) ──────────────────
# Runs a JS snippet in the browser that clicks 'Reconnect' every 60s.
# This prevents Colab from disconnecting while training.
from IPython.display import display, Javascript
display(Javascript('''
  function keepAlive() {
    var reconnect = document.querySelector("colab-toolbar-button#connect");
    if (reconnect && reconnect.shadowRoot) {
      var btn = reconnect.shadowRoot.querySelector("paper-icon-button");
      if (btn) btn.click();
    }
    // Also click run-anyway if a dialog appears
    document.querySelectorAll("paper-button").forEach(btn => {
      if (btn.innerText === 'RUN ANYWAY') btn.click();
    });
  }
  setInterval(keepAlive, 60000);
  console.log('Auto-reconnect enabled — Colab will stay connected.');
'''))
print('✅ Auto-reconnect enabled — Colab will stay connected during training')
print('   Leave this tab open. Training can run for 12h before needing manual reconnect.')


In [ ]:
# ── 0. GPU check ──────────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:', result.stdout.strip())
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → GPU (T4 is free)')

import torch
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q pillow numpy matplotlib requests aiohttp groq anthropic google-generativeai
!pip install -q onnx onnxruntime
# torch is pre-installed on Colab — skip re-install to save time
print('✅ Dependencies ready')

In [ ]:
# ── Inline training modules (no repo clone needed) ────────────────────────────
import os, sys, json
from pathlib import Path

# API keys from Colab Secrets
GROQ_KEY = GEMINI_KEY = ANTHROPIC_KEY = PIXELLAB_KEY = ""
try:
    from google.colab import userdata  # type: ignore
    GROQ_KEY      = userdata.get("GROQ_API_KEY") or ""
    GEMINI_KEY    = userdata.get("GEMINI_API_KEY") or ""
    ANTHROPIC_KEY = userdata.get("ANTHROPIC_API_KEY") or ""
    PIXELLAB_KEY  = userdata.get("PIXELLAB_API_KEY") or ""
    print("✅ Loaded secrets from Colab Secrets")
except Exception:
    print("ℹ️  No Colab Secrets")

os.environ["GROQ_API_KEY"]      = GROQ_KEY
os.environ["GEMINI_API_KEY"]    = GEMINI_KEY
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_KEY
os.environ["PIXELLAB_API_KEY"]  = PIXELLAB_KEY

# Create training_data dir
Path("/content/training_data").mkdir(parents=True, exist_ok=True)
os.chdir("/content")
print("✅ Environment ready")


In [ ]:
# ── 3. PPO Policy — GPU-accelerated ──────────────────────────────────────────
import torch  # type: ignore
import torch.nn as nn  # type: ignore
import torch.optim as optim  # type: ignore
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
import json
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── State encoder (converts score dict → feature vector) ─────────────────────
STATE_DIM  = 16   # art_score, code_score, design_score, level, episode, ...
ACTION_DIM = 8    # one action type per slot
ACTION_NAMES = [
    'draw_character', 'draw_tile', 'draw_prop',
    'generate_scene_code', 'generate_player_code', 'generate_enemy_code',
    'design_map', 'design_game',
]

class PolicyNet(nn.Module):
    """Actor-Critic network for PPO."""
    def __init__(self, state_dim=STATE_DIM, action_dim=ACTION_DIM, hidden=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
        )
        self.actor  = nn.Linear(hidden, action_dim)   # policy logits
        self.critic = nn.Linear(hidden, 1)            # value estimate

    def forward(self, x):
        h = self.shared(x)
        return self.actor(h), self.critic(h)

    def act(self, state: torch.Tensor):
        logits, value = self(state)
        dist   = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action), value

@dataclass
class Trajectory:
    states:      List[torch.Tensor] = field(default_factory=list)
    actions:     List[torch.Tensor] = field(default_factory=list)
    log_probs:   List[torch.Tensor] = field(default_factory=list)
    values:      List[torch.Tensor] = field(default_factory=list)
    rewards:     List[float]        = field(default_factory=list)
    dones:       List[bool]         = field(default_factory=list)

class PPOTrainer:
    """Proximal Policy Optimisation trainer (GPU-accelerated)."""

    def __init__(self, lr=3e-4, gamma=0.99, lam=0.95, clip=0.2,
                 entropy_coef=0.01, value_coef=0.5, epochs=4):
        self.policy  = PolicyNet().to(device)
        self.optim   = optim.Adam(self.policy.parameters(), lr=lr)
        self.gamma   = gamma
        self.lam     = lam
        self.clip    = clip
        self.ent_c   = entropy_coef
        self.val_c   = value_coef
        self.epochs  = epochs
        self.traj    = Trajectory()
        self.step    = 0

        # Try to load existing checkpoint
        ckpt = Path('training_data/ppo_policy.pt')
        if ckpt.exists():
            self.policy.load_state_dict(torch.load(ckpt, map_location=device))
            print(f'✅ Loaded checkpoint from {ckpt}')
        else:
            print('🆕 Starting fresh PPO policy')

    def encode_state(self, scores: Dict, episode: int, level: int) -> torch.Tensor:
        """Convert score dict → normalised feature vector."""
        art    = scores.get('art_score',    0) / 100.0
        code   = scores.get('code_score',   0) / 100.0
        design = scores.get('design_score', 0) / 100.0
        total  = scores.get('total_score',  0) / 100.0
        ep_n   = min(episode / 200.0, 1.0)
        lvl_n  = level / 4.0
        # pad to STATE_DIM
        vec = [art, code, design, total, ep_n, lvl_n] + [0.0] * (STATE_DIM - 6)
        return torch.tensor(vec, dtype=torch.float32, device=device)

    def compute_gae(self) -> Tuple[torch.Tensor, torch.Tensor]:
        """Generalised Advantage Estimation."""
        rewards  = self.traj.rewards
        values   = [v.item() for v in self.traj.values]
        dones    = self.traj.dones
        T        = len(rewards)
        advs     = [0.0] * T
        last_gae = 0.0
        last_val = 0.0

        for t in reversed(range(T)):
            next_val  = last_val if not dones[t] else 0.0
            delta     = rewards[t] + self.gamma * next_val - values[t]
            last_gae  = delta + self.gamma * self.lam * last_gae * (not dones[t])
            advs[t]   = last_gae
            last_val  = values[t]

        advs    = torch.tensor(advs, dtype=torch.float32, device=device)
        returns = advs + torch.tensor(values, dtype=torch.float32, device=device)
        advs    = (advs - advs.mean()) / (advs.std() + 1e-8)
        return advs, returns

    def update(self):
        """Run PPO update on collected trajectory."""
        if len(self.traj.rewards) < 4:
            return {}

        advs, returns = self.compute_gae()
        states    = torch.stack(self.traj.states)
        actions   = torch.stack(self.traj.actions)
        old_lps   = torch.stack(self.traj.log_probs).detach()

        total_loss = total_pg = total_vl = total_ent = 0.0
        for _ in range(self.epochs):
            logits, values = self.policy(states)
            dist    = torch.distributions.Categorical(logits=logits)
            new_lps = dist.log_prob(actions)
            entropy = dist.entropy().mean()

            ratio   = torch.exp(new_lps - old_lps)
            pg_loss = -torch.min(
                ratio * advs,
                ratio.clamp(1 - self.clip, 1 + self.clip) * advs
            ).mean()
            val_loss = 0.5 * (values.squeeze() - returns).pow(2).mean()
            loss     = pg_loss + self.val_c * val_loss - self.ent_c * entropy

            self.optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
            self.optim.step()

            total_loss += loss.item()
            total_pg   += pg_loss.item()
            total_vl   += val_loss.item()
            total_ent  += entropy.item()

        # Clear trajectory
        self.traj = Trajectory()
        n = self.epochs
        return {'loss': total_loss/n, 'pg': total_pg/n, 'vl': total_vl/n, 'ent': total_ent/n}

    def save_checkpoint(self, path='training_data/ppo_policy.pt'):
        Path(path).parent.mkdir(exist_ok=True)
        torch.save(self.policy.state_dict(), path)

    def export_onnx(self, path='training_data/ppo_policy.onnx'):
        Path(path).parent.mkdir(exist_ok=True)
        dummy = torch.zeros(1, STATE_DIM, device=device)
        torch.onnx.export(
            self.policy, dummy, path,
            input_names=['state'], output_names=['logits', 'value'],
            dynamic_axes={'state': {0: 'batch'}},
            opset_version=11
        )
        print(f'✅ ONNX exported to {path}')

ppo = PPOTrainer()
print(f'Policy parameters: {sum(p.numel() for p in ppo.policy.parameters()):,}')

In [ ]:
# ── Inline AI trainer modules ────────────────────────────────────────────────
# GameEvaluator
"""
Game Quality Evaluator
======================
Scores generated games on three axes:
  - Pixel Art Quality  (0–100)
  - Code Quality       (0–100)
  - Game Design        (0–100)

Applies automatic penalties for bad outputs so the RL trainer
knows exactly what to penalise.

Final score = weighted average of three axes.
"""

from __future__ import annotations

import base64
import io
import math
import re
import subprocess
import tempfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

try:
    from PIL import Image
    import numpy as np
    HAS_PIL = True
except ImportError:
    HAS_PIL = False


# ── Palette definitions (for adherence check) ─────────────────────────────────

DB32_PALETTE = [
    (0,0,0),(34,32,52),(69,40,60),(102,57,49),(143,86,59),(223,113,38),
    (217,160,102),(238,195,154),(251,242,54),(153,229,80),(106,190,48),
    (55,148,110),(75,105,47),(82,75,36),(50,60,57),(63,63,116),(48,96,130),
    (91,110,225),(99,155,255),(95,205,228),(203,219,252),(255,255,255),
    (155,173,183),(132,126,135),(105,106,106),(89,86,82),(118,66,138),
    (172,50,50),(217,87,99),(215,123,186),(143,151,74),(138,111,48)
]
PICO8_PALETTE = [
    (0,0,0),(29,43,83),(126,37,83),(0,135,81),(171,82,54),(95,87,79),
    (194,195,199),(255,241,232),(255,0,77),(255,163,0),(255,236,39),
    (0,228,54),(41,173,255),(131,118,156),(255,119,168),(255,204,170)
]

def _nearest_palette_dist(rgb: tuple, palette: list) -> float:
    r, g, b = rgb
    best = float('inf')
    for pr, pg, pb in palette:
        d = (r-pr)**2 + (g-pg)**2 + (b-pb)**2
        if d < best:
            best = d
    return math.sqrt(best)


# ── Score result dataclass ────────────────────────────────────────────────────

@dataclass
class EvalResult:
    pixel_art_score: float = 0.0
    code_score: float = 0.0
    design_score: float = 0.0
    total_score: float = 0.0
    penalties: list[str] = field(default_factory=list)
    bonuses: list[str] = field(default_factory=list)
    breakdown: dict = field(default_factory=dict)

    def summary(self) -> str:
        lines = [
            f"Total: {self.total_score:.1f}/100",
            f"  Pixel Art : {self.pixel_art_score:.1f}/100",
            f"  Code      : {self.code_score:.1f}/100",
            f"  Design    : {self.design_score:.1f}/100",
        ]
        if self.bonuses:
            lines.append("  ✅ " + " | ".join(self.bonuses))
        if self.penalties:
            lines.append("  ❌ " + " | ".join(self.penalties))
        return "\n".join(lines)


# ── Pixel Art Evaluator ───────────────────────────────────────────────────────

class PixelArtEvaluator:
    """
    Scores a single PIL Image on pixel art quality.
    Rewards: palette adherence, contrast, clean edges, appropriate colour count.
    Penalises: fully black/white, noise, too many colours, zero contrast.
    """

    def evaluate(self, image: "Image.Image") -> tuple[float, list[str], list[str]]:
        if not HAS_PIL:
            return 50.0, [], ["PIL not available — using default score"]

        penalties, bonuses = [], []
        score = 0.0

        img = image.convert("RGBA")
        w, h = img.size
        pixels = list(img.getdata())
        rgb_pixels = [(r, g, b) for r, g, b, a in pixels if a > 10]

        if not rgb_pixels:
            return 0.0, ["Image is fully transparent"], []

        # ── 1. Colour count (20 pts) ─────────────────────────────────────────
        unique_colors = set(rgb_pixels)
        n_colors = len(unique_colors)
        if n_colors <= 2:
            score += 0
            penalties.append(f"Only {n_colors} colours (too flat)")
        elif n_colors <= 32:
            score += 20
            bonuses.append(f"Good colour count ({n_colors})")
        elif n_colors <= 64:
            score += 12
        elif n_colors <= 128:
            score += 6
            penalties.append(f"Too many colours ({n_colors}) — not pixel art style")
        else:
            score += 0
            penalties.append(f"Way too many colours ({n_colors}) — not pixel art")

        # ── 2. Palette adherence (20 pts) ────────────────────────────────────
        sample = list(unique_colors)[:50]
        db32_dists = [_nearest_palette_dist(c, DB32_PALETTE) for c in sample]
        pico_dists = [_nearest_palette_dist(c, PICO8_PALETTE) for c in sample]
        avg_db32 = sum(db32_dists) / len(db32_dists) if db32_dists else 999
        avg_pico = sum(pico_dists) / len(pico_dists) if pico_dists else 999
        best_pal_dist = min(avg_db32, avg_pico)
        if best_pal_dist < 20:
            score += 20
            bonuses.append("Excellent palette adherence")
        elif best_pal_dist < 40:
            score += 14
            bonuses.append("Good palette adherence")
        elif best_pal_dist < 70:
            score += 7
        else:
            score += 0
            penalties.append("Colours don't match any known pixel art palette")

        # ── 3. Contrast (20 pts) ─────────────────────────────────────────────
        rs = [r for r, g, b in rgb_pixels]
        gs = [g for r, g, b in rgb_pixels]
        bs = [b for r, g, b in rgb_pixels]
        brightness = [(r*299 + g*587 + b*114) // 1000 for r, g, b in rgb_pixels]
        contrast = max(brightness) - min(brightness)
        if contrast > 180:
            score += 20
            bonuses.append("High contrast — readable")
        elif contrast > 100:
            score += 14
        elif contrast > 50:
            score += 7
        else:
            score += 0
            penalties.append(f"Very low contrast ({contrast}) — unreadable")

        # ── 4. Not blank (20 pts) ────────────────────────────────────────────
        avg_r = sum(rs) / len(rs)
        avg_g = sum(gs) / len(gs)
        avg_b = sum(bs) / len(bs)
        is_all_black = avg_r < 10 and avg_g < 10 and avg_b < 10
        is_all_white = avg_r > 245 and avg_g > 245 and avg_b > 245
        if is_all_black:
            score += 0
            penalties.append("Image is completely black — MAJOR PENALTY")
        elif is_all_white:
            score += 0
            penalties.append("Image is completely white — MAJOR PENALTY")
        else:
            score += 20

        # ── 5. Edge cleanliness (20 pts) ────────────────────────────────────
        # Count isolated single pixels (surrounded by transparency/very different colour)
        if HAS_PIL:
            try:
                import numpy as np
                arr = np.array(img)
                alpha = arr[:, :, 3]
                # Edge = pixels with alpha >10 adjacent to alpha==0
                inner = (alpha[1:-1, 1:-1] > 10)
                top    = (alpha[0:-2, 1:-1] == 0)
                bot    = (alpha[2:,   1:-1] == 0)
                left   = (alpha[1:-1, 0:-2] == 0)
                right  = (alpha[1:-1, 2:]   == 0)
                isolated = inner & top & bot & left & right
                iso_frac = isolated.sum() / max(inner.sum(), 1)
                if iso_frac < 0.02:
                    score += 20
                    bonuses.append("Clean edges")
                elif iso_frac < 0.08:
                    score += 12
                elif iso_frac < 0.20:
                    score += 5
                    penalties.append(f"Noisy edges ({iso_frac:.1%} isolated pixels)")
                else:
                    score += 0
                    penalties.append(f"Very noisy/messy art ({iso_frac:.1%} isolated pixels)")
            except Exception:
                score += 10  # neutral

        return min(score, 100.0), penalties, bonuses


# ── GDScript Code Evaluator ───────────────────────────────────────────────────

class CodeEvaluator:
    """
    Scores GDScript code on correctness and completeness.
    Uses regex heuristics (no Godot binary needed at training time).
    """

    REQUIRED_CALLBACKS = ["_ready", "_process", "_input", "_physics_process", "_unhandled_input"]
    GOOD_PATTERNS = [
        (r"extends\s+\w+", 5, "Proper class extension"),
        (r"class_name\s+\w+", 5, "Named class"),
        (r"signal\s+\w+", 5, "Uses signals"),
        (r"@export", 5, "Uses exports for editor config"),
        (r"CollisionShape|CharacterBody|RigidBody|Area2D", 10, "Has physics nodes"),
        (r"AnimationPlayer|AnimatedSprite", 8, "Has animations"),
        (r"Input\.", 8, "Handles player input"),
        (r"get_tree\(\)|SceneTree", 5, "Uses scene tree"),
        (r"save|load|FileAccess", 5, "Has save/load"),
        (r"score|health|mana|stamina|level", 5, "Has game stats"),
        (r"enum\s+\w+", 5, "Uses enums for state"),
        (r"func _ready", 5, "Has _ready"),
        (r"func _process|func _physics_process", 7, "Has update loop"),
    ]
    BAD_PATTERNS = [
        (r"print\s*\(\s*\)", -3, "Empty print statement"),
        (r"pass\s*\n.*pass\s*\n", -5, "Multiple consecutive pass statements"),
        (r"TODO|FIXME|HACK|XXX", -5, "Unfinished placeholder code"),
        (r"^\s*#.*$\n\s*#.*$\n\s*#.*$\n\s*#.*$", -5, "Wall of comments with no code"),
    ]

    def evaluate(self, code: str) -> tuple[float, list[str], list[str]]:
        penalties, bonuses = [], []
        score = 0.0

        if not code or len(code.strip()) < 10:
            return 0.0, ["Empty or nearly empty code — MAJOR PENALTY"], []

        # Line count sanity
        lines = code.split("\n")
        n_lines = len(lines)
        if n_lines < 5:
            penalties.append(f"Very short code ({n_lines} lines)")
            score -= 20
        elif n_lines >= 30:
            score += 10
            bonuses.append(f"Substantial code ({n_lines} lines)")
        elif n_lines >= 10:
            score += 5

        # Good patterns
        for pattern, pts, label in self.GOOD_PATTERNS:
            if re.search(pattern, code, re.IGNORECASE | re.MULTILINE):
                score += pts
                bonuses.append(label)

        # Bad patterns
        for pattern, pts, label in self.BAD_PATTERNS:
            if re.search(pattern, code, re.IGNORECASE | re.MULTILINE):
                score += pts
                penalties.append(label)

        # Syntax heuristics
        open_brackets = code.count("{") + code.count("(")
        close_brackets = code.count("}") + code.count(")")
        if abs(open_brackets - close_brackets) > 5:
            score -= 15
            penalties.append("Unbalanced brackets — likely syntax error")

        # Indentation check (GDScript is indent-sensitive)
        indented = sum(1 for l in lines if l.startswith("\t") or l.startswith("  "))
        if indented < 2 and n_lines > 5:
            score -= 20
            penalties.append("No indentation — GDScript will fail to parse")

        return max(0.0, min(score, 100.0)), penalties, bonuses


# ── Game Design Evaluator ─────────────────────────────────────────────────────

class DesignEvaluator:
    """
    Scores overall game design from code + metadata.
    Checks for player, enemies, goals, progression, UI.
    """

    DESIGN_CHECKS = [
        (r"player|hero|character|protagonist", 15, "Has player character"),
        (r"enemy|monster|mob|npc|boss", 15, "Has enemies/NPCs"),
        (r"win|victory|complete|finish|goal", 15, "Has win condition"),
        (r"die|death|game.?over|lose|health\s*<=\s*0|hp\s*<=\s*0", 15, "Has lose/death condition"),
        (r"score|xp|experience|level.?up|progress", 10, "Has progression"),
        (r"Label|RichText|HUD|UI|interface", 10, "Has UI elements"),
        (r"map|world|dungeon|room|area|zone|biome", 10, "Has world/map design"),
        (r"item|weapon|armor|potion|loot|drop|inventory", 10, "Has items/inventory"),
    ]
    DESIGN_PENALTIES = [
        (r"^\s*extends Node\s*$", -10, "Extends bare Node — no game structure"),
        (r"^\s*pass\s*$", -5, "Empty script body"),
    ]

    def evaluate(self, code: str, description: str = "") -> tuple[float, list[str], list[str]]:
        penalties, bonuses = [], []
        score = 0.0
        full_text = code + " " + description

        for pattern, pts, label in self.DESIGN_CHECKS:
            if re.search(pattern, full_text, re.IGNORECASE):
                score += pts
                bonuses.append(label)

        for pattern, pts, label in self.DESIGN_PENALTIES:
            if re.search(pattern, code, re.IGNORECASE | re.MULTILINE):
                score += pts
                penalties.append(label)

        return max(0.0, min(score, 100.0)), penalties, bonuses


# ── Main Evaluator (combines all three) ──────────────────────────────────────

class GameEvaluator:
    """
    Master evaluator — call .evaluate() with any combination of:
      - image (PIL Image or base64 string)
      - code (GDScript string)
      - description (str) for design hints
    Returns EvalResult with total score, breakdown, penalties, bonuses.
    """

    WEIGHTS = {"pixel_art": 0.35, "code": 0.35, "design": 0.30}

    def __init__(self):
        self._pixel = PixelArtEvaluator()
        self._code  = CodeEvaluator()
        self._design = DesignEvaluator()

    def evaluate(
        self,
        image: Optional[object] = None,
        code: str = "",
        description: str = "",
    ) -> EvalResult:
        result = EvalResult()
        all_penalties = []
        all_bonuses = []

        has_image = image is not None
        has_code = bool(code)
        has_desc = bool(description)

        # ── Pixel art score ──────────────────────────────────────────────────
        if has_image:
            pil_img = self._to_pil(image)
            if pil_img:
                pa_score, pa_pen, pa_bon = self._pixel.evaluate(pil_img)
                result.pixel_art_score = pa_score
                all_penalties += [f"[Art] {p}" for p in pa_pen]
                all_bonuses   += [f"[Art] {b}" for b in pa_bon]
                result.breakdown["pixel_art"] = pa_score
            else:
                result.pixel_art_score = 50.0
        else:
            result.pixel_art_score = 50.0

        # ── Code score ──────────────────────────────────────────────────────
        if has_code:
            c_score, c_pen, c_bon = self._code.evaluate(code)
            result.code_score = c_score
            all_penalties += [f"[Code] {p}" for p in c_pen]
            all_bonuses   += [f"[Code] {b}" for b in c_bon]
            result.breakdown["code"] = c_score
        else:
            result.code_score = 50.0

        # ── Design score ─────────────────────────────────────────────────────
        if has_code or has_desc:
            d_score, d_pen, d_bon = self._design.evaluate(code, description)
            result.design_score = d_score
            all_penalties += [f"[Design] {p}" for p in d_pen]
            all_bonuses   += [f"[Design] {b}" for b in d_bon]
            result.breakdown["design"] = d_score
        else:
            result.design_score = 50.0

        # ── Adaptive weights (art-only tasks don't get penalised on design) ─
        if has_image and not has_code:
            # Pure art task: weight art heavily, code/design are neutral fillers
            w = {"pixel_art": 0.70, "code": 0.15, "design": 0.15}
        elif has_code and not has_image:
            # Pure code task: weight code + design, art is neutral
            w = {"pixel_art": 0.10, "code": 0.50, "design": 0.40}
        else:
            # Mixed task
            w = self.WEIGHTS

        result.total_score = (
            result.pixel_art_score * w["pixel_art"]
            + result.code_score    * w["code"]
            + result.design_score  * w["design"]
        )

        result.penalties = all_penalties
        result.bonuses   = all_bonuses
        return result

    def _to_pil(self, image) -> Optional["Image.Image"]:
        if not HAS_PIL:
            return None
        if isinstance(image, str):
            # base64 PNG
            try:
                data = base64.b64decode(image)
                return Image.open(io.BytesIO(data))
            except Exception:
                return None
        try:
            return image  # already PIL Image
        except Exception:
            return None


# ExperienceMemory  
"""
Experience Memory
=================
Stores training episodes (state → action → reward).
Best episodes are replayed as few-shot examples to guide future generation.
Worst episodes are used as negative examples ("don't do this").

Persisted to JSON so training survives restarts.
"""

from __future__ import annotations

import json
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional


MEMORY_PATH = Path(__file__).parent.parent / "training_data" / "rl_memory.json"


@dataclass
class Episode:
    episode_id: str
    timestamp: float
    action_type: str          # "draw_character", "generate_code", "design_map", etc.
    action_params: dict       # what was asked
    output_summary: str       # brief description of what was generated
    pixel_art_score: float
    code_score: float
    design_score: float
    total_score: float
    penalties: list[str] = field(default_factory=list)
    bonuses: list[str]   = field(default_factory=list)
    lesson: str = ""          # derived lesson for the agent

    @property
    def is_good(self) -> bool:
        return self.total_score >= 65.0

    @property
    def is_bad(self) -> bool:
        return self.total_score < 40.0


class ExperienceMemory:
    """
    Ring-buffer of training episodes, ranked by score.
    Provides:
      - top_examples(n)   → best n episodes (positive examples)
      - bad_examples(n)   → worst n episodes (negative examples / lessons)
      - few_shot_prompt() → formatted string to inject into LLM context
    """

    MAX_SIZE = 500

    def __init__(self):
        MEMORY_PATH.parent.mkdir(parents=True, exist_ok=True)
        self._episodes: list[Episode] = []
        self._load()

    # ── Persistence ────────────────────────────────────────────────────────────

    def _load(self):
        if MEMORY_PATH.exists():
            try:
                raw = json.loads(MEMORY_PATH.read_text())
                self._episodes = [Episode(**e) for e in raw]
                print(f"[Memory] Loaded {len(self._episodes)} episodes from disk")
            except Exception as e:
                print(f"[Memory] Load failed: {e} — starting fresh")

    def save(self):
        data = [asdict(e) for e in self._episodes]
        MEMORY_PATH.write_text(json.dumps(data, indent=2))

    # ── Add episode ────────────────────────────────────────────────────────────

    def add(
        self,
        action_type: str,
        action_params: dict,
        output_summary: str,
        pixel_art_score: float,
        code_score: float,
        design_score: float,
        total_score: float,
        penalties: list[str],
        bonuses: list[str],
    ) -> Episode:
        lesson = self._derive_lesson(penalties, bonuses, total_score)
        ep = Episode(
            episode_id=f"ep_{int(time.time()*1000)}",
            timestamp=time.time(),
            action_type=action_type,
            action_params=action_params,
            output_summary=output_summary,
            pixel_art_score=pixel_art_score,
            code_score=code_score,
            design_score=design_score,
            total_score=total_score,
            penalties=penalties,
            bonuses=bonuses,
            lesson=lesson,
        )
        self._episodes.append(ep)
        # Keep only MAX_SIZE most recent
        if len(self._episodes) > self.MAX_SIZE:
            self._episodes = sorted(self._episodes, key=lambda e: e.timestamp)[-self.MAX_SIZE:]
        self.save()
        return ep

    # ── Retrieval ──────────────────────────────────────────────────────────────

    def top_examples(self, n: int = 5, action_type: Optional[str] = None) -> list[Episode]:
        eps = self._episodes
        if action_type:
            eps = [e for e in eps if e.action_type == action_type]
        return sorted(eps, key=lambda e: e.total_score, reverse=True)[:n]

    def bad_examples(self, n: int = 5, action_type: Optional[str] = None) -> list[Episode]:
        eps = self._episodes
        if action_type:
            eps = [e for e in eps if e.action_type == action_type]
        return sorted(eps, key=lambda e: e.total_score)[:n]

    def stats(self) -> dict:
        if not self._episodes:
            return {"total": 0, "avg_score": 0, "best": 0, "worst": 0, "good_pct": 0}
        scores = [e.total_score for e in self._episodes]
        good = sum(1 for e in self._episodes if e.is_good)
        return {
            "total": len(self._episodes),
            "avg_score": round(sum(scores) / len(scores), 1),
            "best": round(max(scores), 1),
            "worst": round(min(scores), 1),
            "good_pct": round(good / len(self._episodes) * 100, 1),
        }

    def reward_curve(self) -> list[float]:
        """Returns list of total scores in chronological order (for plotting)."""
        sorted_eps = sorted(self._episodes, key=lambda e: e.timestamp)
        return [e.total_score for e in sorted_eps]

    # ── Few-shot prompt builder ────────────────────────────────────────────────

    def few_shot_prompt(self, action_type: Optional[str] = None) -> str:
        """
        Builds a text block the LLM can read to understand what good/bad looks like.
        Inject this into system or user prompt before generation.
        """
        lines = ["=== TRAINING MEMORY ===",
                 "You have learned from past generation attempts. Apply these lessons:\n"]

        # Positive lessons
        good = self.top_examples(3, action_type)
        if good:
            lines.append("✅ WHAT WORKED WELL:")
            for ep in good:
                lines.append(f"  • [{ep.action_type}] Score {ep.total_score:.0f}/100 — {ep.lesson}")
                if ep.bonuses:
                    lines.append(f"    Bonuses: {', '.join(ep.bonuses[:3])}")

        lines.append("")

        # Negative lessons
        bad = self.bad_examples(3, action_type)
        bad = [b for b in bad if b.is_bad]
        if bad:
            lines.append("❌ WHAT TO AVOID (penalised in past):")
            for ep in bad:
                lines.append(f"  • [{ep.action_type}] Score {ep.total_score:.0f}/100 — {ep.lesson}")
                if ep.penalties:
                    lines.append(f"    Penalties: {', '.join(ep.penalties[:3])}")

        lines.append("\n=== END TRAINING MEMORY ===\n")
        return "\n".join(lines)

    # ── Lesson derivation ─────────────────────────────────────────────────────

    def _derive_lesson(self, penalties: list[str], bonuses: list[str], score: float) -> str:
        if score >= 80:
            top = bonuses[0] if bonuses else "high quality output"
            return f"Excellent output ({score:.0f}/100): {top}"
        elif score >= 60:
            return f"Good output ({score:.0f}/100) — keep similar approach"
        elif score >= 40:
            tip = penalties[0] if penalties else "needs improvement"
            return f"Average output ({score:.0f}/100) — fix: {tip}"
        else:
            tip = penalties[0] if penalties else "major quality issues"
            return f"PENALISED ({score:.0f}/100) — avoid: {tip}"


# RLTrainer
"""
RL Trainer
==========
Reinforcement learning loop for the AI Game Dev agent.

Each episode:
  1. GENERATE — agent produces pixel art / code / design
  2. EVALUATE — GameEvaluator scores the output (0–100)
  3. REWARD / PENALISE — result stored in ExperienceMemory
  4. IMPROVE — next generation uses few_shot_prompt from memory

The agent improves over time by:
  - Seeing its own best examples as positive demonstrations
  - Seeing its own worst examples as "do NOT do this"
  - Gradually increasing the quality threshold to unlock harder tasks

Curriculum:
  Level 1 (score < 40): Basic — single tile, simple character, 20-line script
  Level 2 (score 40–65): Medium — animated character, multi-tile map, full scene
  Level 3 (score 65–80): Hard — full game scene, enemies, items, save/load
  Level 4 (score > 80): Expert — complete playable game, polished art, cutscenes
"""

from __future__ import annotations

import asyncio
import time
import traceback
from dataclasses import dataclass
from typing import Optional, Callable

from .game_evaluator import GameEvaluator, EvalResult
from .experience_memory import ExperienceMemory


# ── Training config ────────────────────────────────────────────────────────────

@dataclass
class TrainingConfig:
    max_episodes: int = 100
    target_score: float = 75.0       # stop when rolling avg hits this
    penalty_threshold: float = 40.0  # below this = penalised episode
    reward_threshold: float = 65.0   # above this = rewarded episode
    window_size: int = 10            # rolling average window
    action_types: list = None        # None = all types

    def __post_init__(self):
        if self.action_types is None:
            self.action_types = [
                "draw_character",
                "draw_tile",
                "draw_prop",
                "generate_scene_code",
                "generate_player_code",
                "generate_enemy_code",
                "design_map",
                "design_game",
            ]


# ── Curriculum ────────────────────────────────────────────────────────────────

CURRICULUM = [
    {
        "level": 1,
        "name": "Basics",
        "min_score": 0,
        "tasks": [
            {
                "action": "draw_character",
                "prompt": "Draw a simple warrior character facing south. Use DB32 palette. Keep it clean with ≤16 colours.",
                "params": {"archetype": "warrior", "direction": "south", "size": 32},
            },
            {
                "action": "draw_tile",
                "prompt": "Draw a grass tile, 16×16 pixels, clean palette, no noise.",
                "params": {"tile_type": "grass", "size": 16},
            },
            {
                "action": "generate_player_code",
                "prompt": "Write minimal GDScript for a CharacterBody2D player with WASD movement and a health variable.",
                "params": {"complexity": "simple"},
            },
        ],
    },
    {
        "level": 2,
        "name": "Intermediate",
        "min_score": 40,
        "tasks": [
            {
                "action": "draw_character",
                "prompt": "Draw a detailed mage character with a staff, 4 directions, DB32 palette, clean edges.",
                "params": {"archetype": "mage", "all_directions": True, "size": 32},
            },
            {
                "action": "generate_scene_code",
                "prompt": "Write GDScript for a game scene with a player, a tilemap, enemy spawner, and score label.",
                "params": {"complexity": "medium"},
            },
            {
                "action": "design_map",
                "prompt": "Design a village map with grass biome, water border, forest patches, and a path.",
                "params": {"biomes": ["grass", "water", "forest"], "size": "medium"},
            },
        ],
    },
    {
        "level": 3,
        "name": "Advanced",
        "min_score": 65,
        "tasks": [
            {
                "action": "generate_enemy_code",
                "prompt": "Write GDScript for an enemy with patrol AI, attack range detection, health, and death animation trigger.",
                "params": {"complexity": "hard"},
            },
            {
                "action": "design_game",
                "prompt": "Design a complete isekai RPG: player stats, inventory, quest system, NPC dialog, and save/load.",
                "params": {"genre": "isekai_rpg", "complexity": "full"},
            },
        ],
    },
    {
        "level": 4,
        "name": "Expert",
        "min_score": 80,
        "tasks": [
            {
                "action": "design_game",
                "prompt": (
                    "Create a complete, polished, playable isekai village sim RPG in the style of Vagabond's art quality. "
                    "Include: animated sprites for all characters, procedural world map, combat system, crafting, "
                    "NPC AI with schedules, quest journal, and atmospheric music triggers."
                ),
                "params": {"genre": "isekai_rpg", "complexity": "expert"},
            },
        ],
    },
]


# ── RL Trainer ────────────────────────────────────────────────────────────────

class RLTrainer:
    """
    Main training orchestrator.
    Runs episodes, scores them, stores results, raises curriculum difficulty.
    """

    def __init__(
        self,
        config: Optional[TrainingConfig] = None,
        on_episode: Optional[Callable] = None,
    ):
        self.config = config or TrainingConfig()
        self.memory = ExperienceMemory()
        self.evaluator = GameEvaluator()
        self.on_episode = on_episode  # callback(episode_result_dict) for streaming to UI

        self._running = False
        self._episode_count = 0
        self._score_history: list[float] = []
        self._current_level = 1
        self._generate_fn: Optional[Callable] = None

    def set_generator(self, fn: Callable):
        """Inject the generation function (calls agent endpoints)."""
        self._generate_fn = fn

    @property
    def rolling_avg(self) -> float:
        if not self._score_history:
            return 0.0
        window = self._score_history[-self.config.window_size:]
        return sum(window) / len(window)

    @property
    def current_curriculum_level(self) -> dict:
        for lvl in reversed(CURRICULUM):
            if self.rolling_avg >= lvl["min_score"]:
                return lvl
        return CURRICULUM[0]

    def stop(self):
        self._running = False

    async def run(self) -> dict:
        """
        Run the full training loop.
        Returns summary stats when done.
        """
        self._running = True
        print(f"\n{'='*60}")
        print("🎮 AI Game Dev — RL Training Starting")
        print(f"   Max episodes : {self.config.max_episodes}")
        print(f"   Target score : {self.config.target_score}")
        print(f"   Penalty below: {self.config.penalty_threshold}")
        print(f"{'='*60}\n")

        for ep_num in range(1, self.config.max_episodes + 1):
            if not self._running:
                print("[Trainer] Stopped by user")
                break

            self._episode_count = ep_num
            curr = self.current_curriculum_level
            task = self._pick_task(curr)

            print(f"\n── Episode {ep_num}/{self.config.max_episodes} "
                  f"[Level {curr['level']}: {curr['name']}] ──")
            print(f"   Task: {task['action']}")
            print(f"   Prompt: {task['prompt'][:80]}...")

            # Add memory context to prompt
            memory_ctx = self.memory.few_shot_prompt(task["action"])
            full_prompt = memory_ctx + "\n" + task["prompt"]

            # Generate
            try:
                output = await self._generate(task, full_prompt)
            except Exception as e:
                print(f"   ⚠️  Generation failed: {e}")
                output = {"error": str(e)}

            # Evaluate
            result = self._evaluate(task["action"], output)

            # Store in memory
            ep = self.memory.add(
                action_type=task["action"],
                action_params=task["params"],
                output_summary=output.get("summary", task["action"]),
                pixel_art_score=result.pixel_art_score,
                code_score=result.code_score,
                design_score=result.design_score,
                total_score=result.total_score,
                penalties=result.penalties,
                bonuses=result.bonuses,
            )

            self._score_history.append(result.total_score)

            # Print episode result
            status = "✅ REWARD" if result.total_score >= self.config.reward_threshold else \
                     "❌ PENALTY" if result.total_score < self.config.penalty_threshold else \
                     "⚠️  OK"
            print(f"   {status} — Score: {result.total_score:.1f}/100  "
                  f"(rolling avg: {self.rolling_avg:.1f})")
            print(f"   {result.summary()}")

            # Fire callback for UI streaming
            if self.on_episode:
                self.on_episode({
                    "episode": ep_num,
                    "level": curr["level"],
                    "level_name": curr["name"],
                    "action": task["action"],
                    "score": result.total_score,
                    "rolling_avg": self.rolling_avg,
                    "penalties": result.penalties,
                    "bonuses": result.bonuses,
                    "status": status,
                })

            # Check if target reached
            if (ep_num >= self.config.window_size
                    and self.rolling_avg >= self.config.target_score):
                print(f"\n🎉 Target score {self.config.target_score} reached! "
                      f"Rolling avg: {self.rolling_avg:.1f}")
                break

            # Small delay between episodes
            await asyncio.sleep(0.5)

        self._running = False
        stats = self.memory.stats()
        stats["episodes_run"] = self._episode_count
        stats["final_rolling_avg"] = round(self.rolling_avg, 1)
        stats["level_reached"] = self.current_curriculum_level["name"]
        return stats

    # ── Internal helpers ──────────────────────────────────────────────────────

    def _pick_task(self, curriculum_level: dict) -> dict:
        import random
        return random.choice(curriculum_level["tasks"])

    async def _generate(self, task: dict, full_prompt: str) -> dict:
        """
        Call the generation function.
        If no real generator set, uses the built-in pixel art / code generators.
        """
        if self._generate_fn:
            return await self._generate_fn(task, full_prompt)
        return await self._builtin_generate(task, full_prompt)

    async def _builtin_generate(self, task: dict, prompt: str) -> dict:
        """Built-in generation using local pixel artist + code templates."""
        action = task["action"]
        params = task.get("params", {})

        if action in ("draw_character", "draw_tile", "draw_prop"):
            try:
                import sys
                from pathlib import Path as _Path
                sys.path.insert(0, str(_Path(__file__).parent.parent.parent))
                from ai_game_agent.tools.pixel_artist import (
                    draw_character, draw_tile, draw_prop, draw_to_base64
                )
                if action == "draw_character":
                    img = draw_character(
                        archetype=params.get("archetype", "warrior"),
                        direction=params.get("direction", "south"),
                        size=params.get("size", 32),
                    )
                elif action == "draw_tile":
                    img = draw_tile(
                        tile_type=params.get("tile_type", "grass"),
                        size=params.get("size", 16),
                    )
                else:
                    img = draw_prop(
                        prop_type=params.get("prop_type", "tree"),
                        size=params.get("size", 32),
                    )
                b64 = draw_to_base64(img)
                return {"image": b64, "summary": f"{action} {params}"}
            except Exception as e:
                return {"error": str(e), "summary": f"generation failed: {e}"}

        elif "code" in action or "design" in action:
            # Generate code template based on action type
            code = self._code_template(action, params)
            return {"code": code, "description": prompt, "summary": action}

        return {"summary": action, "note": "no generator for this action type"}

    def _code_template(self, action: str, params: dict) -> str:
        """Returns a GDScript template appropriate for the action."""
        complexity = params.get("complexity", "simple")
        if "player" in action:
            return _PLAYER_TEMPLATE_HARD if complexity == "hard" else _PLAYER_TEMPLATE_SIMPLE
        elif "enemy" in action:
            return _ENEMY_TEMPLATE
        elif "scene" in action:
            return _SCENE_TEMPLATE
        elif "design_game" in action:
            return _GAME_DESIGN_TEMPLATE
        else:
            return _PLAYER_TEMPLATE_SIMPLE

    def _evaluate(self, action_type: str, output: dict) -> EvalResult:
        image = output.get("image")
        code  = output.get("code", "")
        desc  = output.get("description", "") + " " + output.get("summary", "")
        return self.evaluator.evaluate(image=image, code=code, description=desc)


# ── GDScript templates ─────────────────────────────────────────────────────────
# Used for training when no LLM is available

_PLAYER_TEMPLATE_SIMPLE = """\
extends CharacterBody2D

class_name Player

var speed := 200.0
var health := 100
var score := 0

func _ready() -> void:
\tadd_to_group("player")

func _physics_process(delta: float) -> void:
\tvar dir := Vector2.ZERO
\tif Input.is_action_pressed("ui_right"): dir.x += 1
\tif Input.is_action_pressed("ui_left"):  dir.x -= 1
\tif Input.is_action_pressed("ui_up"):    dir.y -= 1
\tif Input.is_action_pressed("ui_down"):  dir.y += 1
\tvelocity = dir.normalized() * speed
\tmove_and_slide()

func take_damage(amount: int) -> void:
\thealth -= amount
\tif health <= 0:
\t\tdie()

func die() -> void:
\tget_tree().reload_current_scene()
"""

_PLAYER_TEMPLATE_HARD = """\
extends CharacterBody2D

class_name Player

signal health_changed(new_health: int)
signal died

enum State { IDLE, WALK, RUN, ATTACK, HURT, DEAD }

@export var speed := 200.0
@export var sprint_multiplier := 1.6
@export var max_health := 100
@export var attack_damage := 15

var health := max_health:
\tset(v):
\t\thealth = clamp(v, 0, max_health)
\t\thealth_changed.emit(health)
\t\tif health == 0:
\t\t\t_change_state(State.DEAD)

var score := 0
var level := 1
var xp := 0

var _state := State.IDLE
var _facing := Vector2.DOWN

func _ready() -> void:
\tadd_to_group("player")
\t$AnimationPlayer.play("idle")

func _physics_process(delta: float) -> void:
\tif _state == State.DEAD: return
\tvar dir := Input.get_vector("ui_left","ui_right","ui_up","ui_down")
\tvar sprinting := Input.is_action_pressed("sprint")
\tvar spd := speed * (sprint_multiplier if sprinting else 1.0)
\tif dir != Vector2.ZERO:
\t\t_facing = dir
\t\tvelocity = dir.normalized() * spd
\t\t_change_state(State.RUN if sprinting else State.WALK)
\telse:
\t\tvelocity = velocity.move_toward(Vector2.ZERO, spd * 10 * delta)
\t\t_change_state(State.IDLE)
\tmove_and_slide()

func _input(event: InputEvent) -> void:
\tif event.is_action_pressed("attack"):
\t\t_do_attack()

func _do_attack() -> void:
\tif _state in [State.DEAD, State.ATTACK]: return
\t_change_state(State.ATTACK)
\tvar area := $AttackArea
\tfor body in area.get_overlapping_bodies():
\t\tif body.is_in_group("enemy"):
\t\t\tbody.take_damage(attack_damage)

func take_damage(amount: int) -> void:
\tif _state == State.DEAD: return
\thealth -= amount
\tif health > 0:
\t\t_change_state(State.HURT)

func gain_xp(amount: int) -> void:
\txp += amount
\tif xp >= level * 100:
\t\tlevel += 1
\t\txp = 0
\t\tmax_health += 10
\t\thealth = max_health

func _change_state(new_state: State) -> void:
\tif _state == new_state: return
\t_state = new_state
\tmatch _state:
\t\tState.IDLE:   $AnimationPlayer.play("idle")
\t\tState.WALK:   $AnimationPlayer.play("walk")
\t\tState.RUN:    $AnimationPlayer.play("run")
\t\tState.ATTACK: $AnimationPlayer.play("attack")
\t\tState.HURT:   $AnimationPlayer.play("hurt")
\t\tState.DEAD:
\t\t\t$AnimationPlayer.play("death")
\t\t\tdied.emit()
"""

_ENEMY_TEMPLATE = """\
extends CharacterBody2D

class_name Enemy

signal died(position: Vector2)

enum State { PATROL, CHASE, ATTACK, HURT, DEAD }

@export var speed := 80.0
@export var chase_speed := 130.0
@export var health := 50
@export var attack_damage := 8
@export var detection_range := 200.0
@export var attack_range := 40.0
@export var loot_drop_chance := 0.3

var _state := State.PATROL
var _player: Node = null
var _patrol_dir := Vector2.RIGHT
var _attack_timer := 0.0

func _ready() -> void:
\tadd_to_group("enemy")
\t_player = get_tree().get_first_node_in_group("player")

func _physics_process(delta: float) -> void:
\tif _state == State.DEAD: return
\t_attack_timer = max(0.0, _attack_timer - delta)
\tmatch _state:
\t\tState.PATROL: _patrol(delta)
\t\tState.CHASE:  _chase(delta)
\t\tState.ATTACK: _try_attack()
\tmove_and_slide()

func _patrol(delta: float) -> void:
\tvelocity = _patrol_dir * speed
\tif _player and global_position.distance_to(_player.global_position) < detection_range:
\t\t_change_state(State.CHASE)
\tif is_on_wall():
\t\t_patrol_dir *= -1

func _chase(delta: float) -> void:
\tif not _player: return
\tvar dist := global_position.distance_to(_player.global_position)
\tif dist > detection_range * 1.5:
\t\t_change_state(State.PATROL)
\telif dist < attack_range:
\t\t_change_state(State.ATTACK)
\telse:
\t\tvelocity = (_player.global_position - global_position).normalized() * chase_speed

func _try_attack() -> void:
\tif _attack_timer > 0: return
\t_attack_timer = 1.5
\tif _player:
\t\t_player.take_damage(attack_damage)

func take_damage(amount: int) -> void:
\tif _state == State.DEAD: return
\thealth -= amount
\tif health <= 0:
\t\t_die()
\telse:
\t\t_change_state(State.HURT)

func _die() -> void:
\t_change_state(State.DEAD)
\tdied.emit(global_position)
\tif randf() < loot_drop_chance:
\t\t_drop_loot()
\tqueue_free()

func _drop_loot() -> void:
\tpass  # Spawn item scene at global_position

func _change_state(new_state: State) -> void:
\t_state = new_state
"""

_SCENE_TEMPLATE = """\
extends Node2D

class_name GameScene

@onready var player := $Player
@onready var tilemap := $TileMap
@onready var enemy_spawner := $EnemySpawner
@onready var score_label: Label = $HUD/ScoreLabel
@onready var health_bar := $HUD/HealthBar

var score := 0
var wave := 1

func _ready() -> void:
\tplayer.health_changed.connect(_on_player_health_changed)
\tplayer.died.connect(_on_player_died)
\t_start_wave()

func _start_wave() -> void:
\tenemy_spawner.spawn_wave(wave)
\twait_for_enemies_cleared()

func _on_player_health_changed(new_health: int) -> void:
\thealth_bar.value = new_health

func _on_player_died() -> void:
\t$GameOverScreen.show()
\tget_tree().paused = true

func add_score(amount: int) -> void:
\tscore += amount
\tscore_label.text = "Score: %d" % score

func _on_wave_cleared() -> void:
\twave += 1
\t_start_wave()

func wait_for_enemies_cleared() -> void:
\twhile get_tree().get_nodes_in_group("enemy").size() > 0:
\t\tawait get_tree().process_frame
\t_on_wave_cleared()
"""

_GAME_DESIGN_TEMPLATE = """\
# GAME DESIGN — Isekai Village Sim RPG
# Full game architecture with all major systems

extends Node

class_name GameManager

signal quest_completed(quest_id: String)
signal item_obtained(item_name: String, amount: int)
signal level_up(new_level: int)

# ── Player Data ──────────────────────────────────────────────────────────────
var player_name := "Traveller"
var player_class := "Villager"
var level := 1
var xp := 0
var xp_to_next := 100
var gold := 50

# ── Stats ───────────────────────────────────────────────────────────────────
var stats := {
\t"hp": 100, "max_hp": 100,
\t"mp": 50,  "max_mp": 50,
\t"attack": 10, "defense": 5,
\t"speed": 8, "luck": 3
}

# ── Inventory ────────────────────────────────────────────────────────────────
var inventory: Dictionary = {}  # item_id → count
var equipment: Dictionary = {"weapon": "", "armor": "", "accessory": ""}

# ── Quests ───────────────────────────────────────────────────────────────────
var active_quests: Dictionary = {}
var completed_quests: Array[String] = []

# ── World State ──────────────────────────────────────────────────────────────
var current_map := "village"
var discovered_maps: Array[String] = ["village"]
var npc_states: Dictionary = {}
var day := 1
var time_of_day := 0.0  # 0.0 = dawn, 1.0 = night

func _ready() -> void:
\tload_game()
\t_start_day_cycle()

func _process(delta: float) -> void:
\ttime_of_day += delta / 600.0  # 10-minute day
\tif time_of_day >= 1.0:
\t\ttime_of_day = 0.0
\t\t_advance_day()

func gain_xp(amount: int) -> void:
\txp += amount
\tif xp >= xp_to_next:
\t\t_level_up()

func _level_up() -> void:
\tlevel += 1
\txp -= xp_to_next
\txp_to_next = int(xp_to_next * 1.4)
\tstats["max_hp"] += 10
\tstats["hp"] = stats["max_hp"]
\tstats["attack"] += 2
\tlevel_up.emit(level)

func add_item(item_id: String, amount: int = 1) -> void:
\tinventory[item_id] = inventory.get(item_id, 0) + amount
\titem_obtained.emit(item_id, amount)

func save_game() -> void:
\tvar data := {
\t\t"level": level, "xp": xp, "gold": gold,
\t\t"stats": stats, "inventory": inventory,
\t\t"active_quests": active_quests,
\t\t"completed_quests": completed_quests,
\t\t"current_map": current_map, "day": day
\t}
\tvar file := FileAccess.open("user://save.json", FileAccess.WRITE)
\tfile.store_string(JSON.stringify(data))
\tfile.close()

func load_game() -> void:
\tif not FileAccess.file_exists("user://save.json"): return
\tvar file := FileAccess.open("user://save.json", FileAccess.READ)
\tvar data: Dictionary = JSON.parse_string(file.get_as_text())
\tfile.close()
\tlevel = data.get("level", 1)
\txp = data.get("xp", 0)
\tgold = data.get("gold", 50)
\tstats = data.get("stats", stats)
\tinventory = data.get("inventory", {})
\tcompleted_quests = data.get("completed_quests", [])
\tcurrent_map = data.get("current_map", "village")
\tday = data.get("day", 1)

func _advance_day() -> void:
\tday += 1

func _start_day_cycle() -> void:
\tpass
"""


# ── Convenience runner ────────────────────────────────────────────────────────

async def run_training_session(
    episodes: int = 50,
    target: float = 75.0,
    on_episode: Optional[Callable] = None,
) -> dict:
    cfg = TrainingConfig(max_episodes=episodes, target_score=target)
    trainer = RLTrainer(config=cfg, on_episode=on_episode)
    return await trainer.run()


from pathlib import Path
memory = ExperienceMemory()
evaluator = GameEvaluator()

mem_file = Path("/content/training_data/experience_memory.json")
if mem_file.exists():
    memory.load(str(mem_file))
    print(f"✅ Loaded {len(memory.episodes)} prior episodes")

rl = RLTrainer(TrainingConfig(max_episodes=1, target_score=99))
print("✅ Trainers ready")


In [ ]:
# -- 5. Self-restarting training loop (runs forever, resumes on reconnect) ----
# To STOP: run the Stop cell below, or Interrupt kernel (Runtime -> Interrupt).
# On reconnect, re-run from Cell 3 onward -- it will resume from last checkpoint.
# ----------------------------------------------------------------------------- 
import asyncio, json, time
# RLTrainer already inlined above
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import clear_output

DATA_DIR      = Path('/content/training_data')
DATA_DIR.mkdir(exist_ok=True)
SESSION_FILE  = DATA_DIR / 'colab_session.json'
STOP_FLAG     = DATA_DIR / 'stop_colab.flag'
CKPT_PT       = DATA_DIR / 'ppo_policy.pt'
CKPT_ONNX     = DATA_DIR / 'ppo_policy.onnx'

EPISODES_PER_BATCH = 50
SAVE_EVERY         = 10
PUSH_EVERY         = 50
UPDATE_EVERY       = 10
CHART_EVERY        = 5

CURRICULUM = [
    {'level': 1, 'min_score': 0,  'name': 'Basics',       'action': 'draw_character'},
    {'level': 2, 'min_score': 40, 'name': 'Intermediate', 'action': 'generate_scene_code'},
    {'level': 3, 'min_score': 65, 'name': 'Advanced',     'action': 'draw_character'},
    {'level': 4, 'min_score': 80, 'name': 'Expert',       'action': 'design_game'},
]

def load_session():
    if SESSION_FILE.exists():
        s = json.loads(SESSION_FILE.read_text())
        print(f'Resuming from ep {s["total_episodes"]} | avg={s["last_avg10"]:.1f} | lvl={s["curriculum_lvl"]}')
        return s
    return {'total_episodes': 0, 'last_avg10': 0, 'curriculum_lvl': 1,
            'scores_hist': [], 'reward_hist': [], 'loss_hist': []}

def save_session(ep_total, avg10, lvl, scores, rewards, losses):
    SESSION_FILE.write_text(json.dumps({
        'total_episodes': ep_total,
        'last_avg10':     float(avg10),
        'curriculum_lvl': lvl,
        'scores_hist':    [float(x) for x in scores[-500:]],
        'reward_hist':    [float(x) for x in rewards[-500:]],
        'loss_hist':      [float(x) for x in losses[-200:]],
        'saved_at':       time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }, indent=2))

def push_to_github(ep_total):
    pass  # No git push needed - checkpoints saved locally

def get_reward(score, prev_avg):
    return max(0.0, (score - prev_avg) / 100.0) * 2.0 + (score / 100.0) * 0.5 + (-0.5 if score < 30 else 0.0)

async def run_episode(ep, level):
    captured: dict = {}
    def _on_ep(data: dict) -> None:
        captured.update(data)
    try:
        one = RLTrainer(TrainingConfig(max_episodes=1, target_score=99), on_episode=_on_ep)
        await asyncio.wait_for(one.run(), timeout=120)
        s = float(captured.get('score', 50))
        scores: dict = {'art_score': s, 'code_score': s, 'design_score': s}
    except Exception as e:
        scores = {'art_score': 30.0, 'code_score': 30.0, 'design_score': 30.0, 'error': str(e)[:80]}
    scores['total_score'] = float(scores.get('art_score',0)*0.4 + scores.get('code_score',0)*0.3 + scores.get('design_score',0)*0.3)
    return scores

def draw_chart(scores, rewards, losses, ep_total, lvl, elapsed):
    clear_output(wait=True)
    avg10 = float(np.mean(scores[-10:])) if scores else 0
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(scores, 'b-', alpha=0.5)
    if len(scores) >= 10:
        ma = np.convolve(scores, np.ones(10)/10, mode='valid')
        axes[0].plot(range(9, len(scores)), ma, 'r-', lw=2, label='MA-10')
    for thr, col, lbl in [(40,'orange','Lvl2'),(65,'green','Lvl3'),(80,'purple','DIVINE')]:
        axes[0].axhline(thr, ls='--', c=col, alpha=0.7, label=lbl)
    axes[0].set_title(f'Score ep={ep_total} | lvl={lvl} | avg={avg10:.1f}')
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 100)
    axes[1].plot(rewards, 'g-', alpha=0.6)
    axes[1].axhline(0, ls='--', c='grey')
    axes[1].set_title('PPO Reward')
    if losses:
        axes[2].plot(losses, 'r-')
        axes[2].set_title('Policy Loss')
    plt.suptitle(f'AI Dev Continuous GPU Training | {elapsed:.1f}min | {device}', fontsize=12)
    plt.tight_layout(); plt.show()

# -- Main infinite loop -------------------------------------------------------
sess           = load_session()
ep_total       = sess['total_episodes']
curriculum_lvl = sess['curriculum_lvl']
scores_hist    = sess['scores_hist']
reward_hist    = sess['reward_hist']
loss_hist      = sess['loss_hist']
avg10          = sess['last_avg10']
run_start      = time.time()
elapsed        = 0.0

if STOP_FLAG.exists():
    STOP_FLAG.unlink()
    print('Cleared previous stop flag')

print(f'Continuous training started on {device.upper()}')
print(f'Resuming from global episode {ep_total}')
print('To stop: run the Stop cell, or use Runtime -> Interrupt')

try:
    while True:
        for batch_ep in range(EPISODES_PER_BATCH):
            if STOP_FLAG.exists():
                print('Stop flag detected -- saving and exiting')
                raise StopIteration

            ep_total += 1
            elapsed   = (time.time() - run_start) / 60

            avg10 = float(np.mean(scores_hist[-10:])) if scores_hist else 0
            for c in CURRICULUM:
                if avg10 >= c['min_score']:
                    curriculum_lvl = c['level']

            state = ppo.encode_state({'art_score': avg10, 'total_score': avg10}, ep_total, curriculum_lvl)
            action, log_prob, value = ppo.policy.act(state)

            scores = asyncio.get_event_loop().run_until_complete(run_episode(ep_total, curriculum_lvl))
            total  = scores['total_score']
            reward = get_reward(float(total), float(avg10))

            ppo.traj.states.append(state)
            ppo.traj.actions.append(action)
            ppo.traj.log_probs.append(log_prob)
            ppo.traj.values.append(value.squeeze())
            ppo.traj.rewards.append(reward)
            ppo.traj.dones.append(False)

            scores_hist.append(total)
            reward_hist.append(reward)

            if ep_total % UPDATE_EVERY == 0:
                metrics = ppo.update()
                if metrics:
                    loss_hist.append(metrics.get('loss', 0))

            if ep_total % SAVE_EVERY == 0:
                ppo.save_checkpoint(str(CKPT_PT))
                save_session(ep_total, avg10, curriculum_lvl, scores_hist, reward_hist, loss_hist)

            if ep_total % PUSH_EVERY == 0:
                ppo.export_onnx(str(CKPT_ONNX))
                push_to_github(ep_total)

            if ep_total % CHART_EVERY == 0:
                draw_chart(scores_hist, reward_hist, loss_hist, ep_total, curriculum_lvl, elapsed)
                print(f'ep={ep_total:5d} | score={total:.1f} | reward={reward:+.3f} | lvl={curriculum_lvl} | {elapsed:.1f}min')

        ppo.save_checkpoint(str(CKPT_PT))
        save_session(ep_total, avg10, curriculum_lvl, scores_hist, reward_hist, loss_hist)
        print(f'Batch done -- ep={ep_total} | avg10={avg10:.1f} | level={curriculum_lvl} | {elapsed:.1f}min')

except (StopIteration, KeyboardInterrupt):
    pass

ppo.save_checkpoint(str(CKPT_PT))
ppo.export_onnx(str(CKPT_ONNX))
save_session(ep_total, avg10, curriculum_lvl, scores_hist, reward_hist, loss_hist)
push_to_github(ep_total)
draw_chart(scores_hist, reward_hist, loss_hist, ep_total, curriculum_lvl, elapsed)
final_avg = float(np.mean(scores_hist[-10:])) if scores_hist else 0
print(f'Saved at ep={ep_total} | avg10={final_avg:.1f}')
print('Re-run this cell to continue from where you left off.')


In [ ]:
# -- 6. Analysis chart (run any time to inspect training progress) -----------
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

sf = Path('training_data/colab_session.json')
if not sf.exists():
    print('No session data yet -- run the training loop first.')
else:
    s       = json.loads(sf.read_text())
    scores  = s['scores_hist']
    rewards = s['reward_hist']
    losses  = s['loss_hist']
    ep_total= s['total_episodes']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes[0,0].plot(scores, alpha=0.5)
    if len(scores) >= 10:
        ma = np.convolve(scores, np.ones(10)/10, mode='valid')
        axes[0,0].plot(range(9, len(scores)), ma, 'r-', lw=2, label='MA-10')
    for thr, col, lbl in [(40,'orange','Lvl2'),(65,'green','Lvl3'),(80,'purple','DIVINE')]:
        axes[0,0].axhline(thr, ls='--', c=col, alpha=0.7, label=lbl)
    axes[0,0].set_title(f'Score trajectory (ep={ep_total})')
    axes[0,0].legend(); axes[0,0].set_ylim(0,100)
    axes[0,1].plot(rewards, 'g-', alpha=0.6)
    axes[0,1].axhline(0, ls='--', c='grey')
    axes[0,1].set_title('PPO reward signal')
    axes[1,0].hist(scores, bins=20, color='steelblue', edgecolor='white')
    axes[1,0].axvline(np.mean(scores), c='red', label=f'Mean={np.mean(scores):.1f}')
    axes[1,0].set_title('Score distribution'); axes[1,0].legend()
    if losses:
        axes[1,1].plot(losses, 'r-'); axes[1,1].set_title('PPO policy loss')
    plt.suptitle(f'AI Dev -- ep={ep_total} | avg10={s["last_avg10"]:.1f} | level={s["curriculum_lvl"]}', fontsize=13)
    plt.tight_layout()
    plt.savefig('training_data/colab_training_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: training_data/colab_training_summary.png')


In [ ]:
# -- 7. Stop training gracefully ----------------------------------------------
# Run this cell to stop the training loop (it saves before exiting).
from pathlib import Path
Path('training_data/stop_colab.flag').touch()
print('Stop flag created -- training will save and exit after the current episode.')
